# Phase 2 analysis (Agents 3 + 4), run from Google Drive. CPU only

This notebook re-runs the full Phase-2 analysis with the exact code of the project:
- **Agent 3**: `phase2/03_experiments/run_all.sh`. It covers Colab ingest + sanity gates, the W6 BM25 LODO grid, the W1/W2 query-time features, W7 effect sizes, the W3 gap model, W1 QPP + routers and W9 pooling bias, and writes `summary_tables.md`.
- **Agent 4**: `phase2/04_improvements/run_all4.sh`. It covers the alpha grid, E1 per-query alpha, the E4 diagnostics, the E2/E3 cross-encoder evaluation and `summary4.md`.

All outputs (CSVs, figures, logs) are written to `MyDrive/IR_Lexical_Gap/project/phase2/03_experiments/{results,figures,logs,cache}` and `.../04_improvements/{...}`. The figures for the paper are therefore regenerated in Drive.

**Prerequisites. Run these notebooks first, all with Drive mounted:**
1. `00_SETUP_drive`: the datasets.
2. `phase1_full_pipeline_drive`: it runs `lexical_gap_full_pipeline` and writes the Phase-1 results and rankings into `project/results` and `project/cache/runs`.
3. `phase2_dense_runs`: MiniLM / BGE-s / BGE-b rankings in `MyDrive/IR_Lexical_Gap/colab_runs`.
4. `phase2_ce_rerank`: cross-encoder scores in `colab_runs/ce`.

Steps 3 and 4 are optional. Without them, the scripts skip the models and datasets whose inputs are missing; they discover inputs from the files present. The numbers then cover fewer models than `RESULTS_SUMMARY.md`. The prerequisite check below tells you what is missing.

**Runtime: CPU is enough.** Do not waste GPU quota on this notebook.

**Honest limits:**
- **RAM.** The code was developed and run on a 15 GB RAM / 4-core machine. Free Colab has about 12.7 GB RAM and 2 cores. The heaviest steps load Quora's and Touche's top-1000 rankings: each Quora run pickle is about 160 MB on disk and more than 1 GB in RAM. We did **not** measure their peak RAM in Colab.
  - If the session crashes with *"Your session crashed after using all available RAM"*, re-run the notebook. Some finished steps are cached: the W6 BM25 grid, the QPP feature files and the alpha grid are skipped when their files exist. The routers re-run, at about 5 min each.
  - If it crashes again at the same step, use a High-RAM runtime (Colab Pro) or run the two shell scripts on a 16 GB machine. Do not drop datasets: the verdicts assume all 8.
- **Time.** On 2 cores, expect roughly **2-4 h**. The W6 BM25 grid on Quora alone took 53 min on 4 cores. Keep the tab open; free Colab disconnects idle sessions.
- **Numbers.** The Colab run uses rankings that you regenerated yourself. MiniLM/hybrid runs then exist for all 8 datasets, not only the 5 local ones. Tables can therefore differ slightly from `RESULTS_SUMMARY.md` and `04_improvements/RESULTS.md`. The narrative `.md` files are not regenerated; only the CSV/PNG/summary tables are.

## 0. Mount Drive, link the project layout, install packages

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install PyStemmer wordfreq statsmodels
!nproc; free -g

In [ ]:
import shutil
shutil.copy('/content/drive/MyDrive/IR_Lexical_Gap/02_code/phase2/setup_drive.py', '/content/setup_drive.py')  # Drive copy of phase2/colab/setup_drive.py

In [ ]:
!python /content/setup_drive.py link
%cd /content/proj

## 1. Code: copied from Drive `02_code/` into the local project layout
These are the exact project files (`src/` = Phase-1 modules, read-only; `02_code/phase2/` = Agent 3; `02_code/phase2/improvements/` = Agent 4). Their md5 sums are checked against the versions this notebook was built for. A mismatch means somebody edited the Drive copy: the run still works, but say so in the paper's reproducibility note.

In [ ]:
import hashlib, os, shutil
C = '/content/drive/MyDrive/IR_Lexical_Gap/02_code'
COPY = [
[
"src/analysis.py",
"src/analysis.py"
],
[
"src/build_report.py",
"src/build_report.py"
],
[
"src/common.py",
"src/common.py"
],
[
"src/failures.py",
"src/failures.py"
],
[
"src/features.py",
"src/features.py"
],
[
"src/run_dense.py",
"src/run_dense.py"
],
[
"src/run_lexical.py",
"src/run_lexical.py"
],
[
"src/validate_metrics.py",
"src/validate_metrics.py"
],
[
"phase2/bm25_lodo.py",
"phase2/03_experiments/bm25_lodo.py"
],
[
"phase2/colab_ingest.py",
"phase2/03_experiments/colab_ingest.py"
],
[
"phase2/common2.py",
"phase2/03_experiments/common2.py"
],
[
"phase2/effect_sizes.py",
"phase2/03_experiments/effect_sizes.py"
],
[
"phase2/fig_winshare.py",
"phase2/03_experiments/fig_winshare.py"
],
[
"phase2/gap_model.py",
"phase2/03_experiments/gap_model.py"
],
[
"phase2/pooling_bias.py",
"phase2/03_experiments/pooling_bias.py"
],
[
"phase2/qpp_eval.py",
"phase2/03_experiments/qpp_eval.py"
],
[
"phase2/qpp_features.py",
"phase2/03_experiments/qpp_features.py"
],
[
"phase2/router.py",
"phase2/03_experiments/router.py"
],
[
"phase2/summarize.py",
"phase2/03_experiments/summarize.py"
],
[
"phase2/run_all.sh",
"phase2/03_experiments/run_all.sh"
],
[
"phase2/improvements/alpha_grid.py",
"phase2/04_improvements/alpha_grid.py"
],
[
"phase2/improvements/ce_eval.py",
"phase2/04_improvements/ce_eval.py"
],
[
"phase2/improvements/ce_score.py",
"phase2/04_improvements/ce_score.py"
],
[
"phase2/improvements/e1_alpha_router.py",
"phase2/04_improvements/e1_alpha_router.py"
],
[
"phase2/improvements/e4_why_routing_fails.py",
"phase2/04_improvements/e4_why_routing_fails.py"
],
[
"phase2/improvements/summarize4.py",
"phase2/04_improvements/summarize4.py"
],
[
"phase2/improvements/run_all4.sh",
"phase2/04_improvements/run_all4.sh"
]
]
MD5 = {
"src/analysis.py": "501602e7ff21fb6a7541a02ac6ce168c",
"src/build_report.py": "b8478ccbd2d9b6e70ae34bdeb9f77d11",
"src/common.py": "ece13b48a0b458258753d3152f02369a",
"src/failures.py": "5de1e04102722fde3400619d45eb07c9",
"src/features.py": "95688e800c4f35375f2a1f2a41e77d6f",
"src/run_dense.py": "74bb20fd272772f4ee09759db5d34eb9",
"src/run_lexical.py": "ab2853683bada77955192be8084ba133",
"src/validate_metrics.py": "4a0afb0be3e34b1cd194bb9bce00ce32",
"phase2/03_experiments/bm25_lodo.py": "36bf62ada4186f979341aae32ad4fb64",
"phase2/03_experiments/colab_ingest.py": "822f3b8ff251d5e4323ca4cfea478c87",
"phase2/03_experiments/common2.py": "a50f353b039aa927dedc35a504a974d6",
"phase2/03_experiments/effect_sizes.py": "6ec9f635380a0841f69be4342a9274c1",
"phase2/03_experiments/fig_winshare.py": "85b8707163888293a5dbbd8a23b68d2d",
"phase2/03_experiments/gap_model.py": "630b0ad1a3351dfdb223f7f17f0e2b00",
"phase2/03_experiments/pooling_bias.py": "bade1effcaccffefa2b17ead26e1649e",
"phase2/03_experiments/qpp_eval.py": "ea708dad5985fbc3b9336d6f8e451a56",
"phase2/03_experiments/qpp_features.py": "c11cde13ca4de2b09c1f4f312ff02d2a",
"phase2/03_experiments/router.py": "c3cf666d3bc5986c35aa821477ba889e",
"phase2/03_experiments/summarize.py": "2dfa1239e0215fe9bdbf69c778f58f59",
"phase2/03_experiments/run_all.sh": "06d0a9693e2b5c06dc419b65d3e6ff6e",
"phase2/04_improvements/alpha_grid.py": "c4077651d11b3905a4a9d83ede6aac9a",
"phase2/04_improvements/ce_eval.py": "f221c502416cd4e875aa8db5464a095e",
"phase2/04_improvements/ce_score.py": "1ff26922df87c2d81ccff68cfd6783ff",
"phase2/04_improvements/e1_alpha_router.py": "0c0857a13f1607df3eb20fdd7e642de6",
"phase2/04_improvements/e4_why_routing_fails.py": "e3ac7816dfd00dbf382ae1aec22b2064",
"phase2/04_improvements/summarize4.py": "01923c4b0ce5ec18f77c866db55f4909",
"phase2/04_improvements/run_all4.sh": "65975ac9e73b991358ad878ea5b84263"
}
bad = []
for src, dst in COPY:
    os.makedirs(os.path.dirname('/content/proj/' + dst), exist_ok=True)
    shutil.copy(f'{C}/{src}', '/content/proj/' + dst)
    h = hashlib.md5(open('/content/proj/' + dst, 'rb').read()).hexdigest()
    if h != MD5[dst]:
        bad.append(dst)
print(len(COPY), 'files copied;', 'all md5 match the built version' if not bad else f'CHANGED on Drive: {bad}')

## 2. Prerequisite check

In [ ]:
import glob, os
os.chdir('/content/proj')
DS = ['nfcorpus', 'scifact', 'fiqa', 'scidocs', 'arguana', 'trec-covid', 'quora', 'webis-touche2020']
def have(pattern): return sorted(os.path.basename(p) for p in glob.glob(pattern))
missing = []
for d in DS:
    row = {'data': os.path.exists(f'data/{d}/corpus.jsonl'),
           'perquery_lexical': os.path.exists(f'results/perquery_lexical_{d}.csv'),
           'features': os.path.exists(f'results/features_{d}.csv'),
           'bm25 run': os.path.exists(f'cache/runs/{d}__bm25.pkl'),
           'hybrid_cc_minilm run': os.path.exists(f'cache/runs/{d}__hybrid_cc_minilm.pkl'),
           'colab dense npz': len(have(f'phase2/03_experiments/colab_runs/run_*_{d}.npz')),
           'colab CE': os.path.exists(f'phase2/03_experiments/colab_runs/ce/ce_{d}.pkl')}
    print(f'{d:18s}', '  '.join(f'{k}={v}' for k, v in row.items()))
    missing += [f'{d}: {k}' for k in ('data', 'perquery_lexical', 'features', 'bm25 run') if not row[k]]
if missing:
    print('\nREQUIRED INPUTS MISSING (run 00_SETUP_drive and phase1_full_pipeline_drive first):')
    print('\n'.join(missing))
else:
    print('\nall required Phase-1 inputs present')

## 3. Agent 3 pipeline (`run_all.sh`)
The log of each step is in `phase2/03_experiments/logs/` on Drive. `COLAB_RUNS` points at `MyDrive/IR_Lexical_Gap/colab_runs`.

In [ ]:
%%bash
export COLAB_RUNS=/content/drive/MyDrive/IR_Lexical_Gap/colab_runs
cd /content/proj && bash phase2/03_experiments/run_all.sh 2>&1 | tail -60

## 4. Agent 4 pipeline (`run_all4.sh`)

In [ ]:
%%bash
export COLAB_RUNS=/content/drive/MyDrive/IR_Lexical_Gap/colab_runs
cd /content/proj && bash phase2/04_improvements/run_all4.sh 2>&1 | tail -60

## 5. Results and figures (already saved in Drive)

In [ ]:
import glob
from IPython.display import Image, Markdown, display
for f in ['phase2/03_experiments/results/summary_tables.md', 'phase2/04_improvements/results/summary4.md']:
    if os.path.exists(f):
        display(Markdown(open(f).read()))
for f in sorted(glob.glob('phase2/03_experiments/figures/*.png')) + sorted(glob.glob('phase2/04_improvements/figures/*.png')):
    print(f); display(Image(f, width=800))

## 6. Mirror the code into the Drive project and flush Drive

In [ ]:
import shutil
D = '/content/drive/MyDrive/IR_Lexical_Gap/project'
for pat in ['src/*.py', 'phase2/03_experiments/*.py', 'phase2/03_experiments/*.sh',
            'phase2/04_improvements/*.py', 'phase2/04_improvements/*.sh']:
    for f in glob.glob(pat):
        os.makedirs(os.path.dirname(f'{D}/{f}'), exist_ok=True)
        shutil.copy(f, f'{D}/{f}')
from google.colab import drive
drive.flush_and_unmount()
print('Drive flushed. Outputs are in MyDrive/IR_Lexical_Gap/project/phase2/.')